# Random Forest — Out-of-Bag (OOB) Score

Each tree in a Random Forest is trained on a **bootstrap sample** — a random draw with replacement from the training set. On average, roughly **37 % of the original training points are never selected** for a given bootstrap draw. These leftover samples are called **Out-of-Bag (OOB)** samples.

Since a given tree has never seen its OOB samples during training, those samples act as a **built-in validation set** for that tree. The OOB score is computed by:
1. For each training sample, collecting predictions only from the trees whose bootstrap draw did **not** include that sample.
2. Aggregating those predictions (majority vote for classification) to produce one OOB prediction per training sample.
3. Comparing all OOB predictions against the true labels to get an accuracy estimate.

The OOB score is an approximately unbiased estimate of the model's generalisation error — similar to cross-validation, but computed for free during training without any additional data split.

In [43]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

 ### Heart Disease Dataset

The **Heart Disease UCI dataset** contains clinical attributes for 303 patients and a binary `target` label indicating the presence (1) or absence (0) of heart disease.


In [20]:
data = pd.read_csv("heart.csv")
data.head(3)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1


### Quick Data Check

Before training,  verify the dataset shape and check for missing values.

In [23]:
print("Shape  :", data.shape)
print("\nMissing values per column:")
print(data.isnull().sum())
print("\nClass distribution:")
print(data['target'].value_counts())

Shape  : (303, 14)

Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Class distribution:
target
1    165
0    138
Name: count, dtype: int64


### Features, Labels, and Train/Test Split

We separate the dataset into:
- `X` — all columns except `target` (the input features)
- `y` — the `target` column (the label to predict)

A standard 80/20 train-test split with a fixed `random_state` ensures reproducibility.

In [26]:
X = data.drop(columns=["target"])
y = data['target']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

### Train Random Forest with OOB Score Enabled

Setting `oob_score=True` instructs scikit-learn to maintain bookkeeping during training so that OOB predictions can be computed for every training sample. This adds negligible overhead and requires no extra data.

Default hyperparameters used:
- `n_estimators=100` — 100 trees (scikit-learn default)
- `max_features='sqrt'` — at each split, `sqrt(n_features)` features are randomly drawn (scikit-learn default for classifiers)

In [29]:
random_forest = RandomForestClassifier(oob_score=True)
random_forest.fit(X_train,y_train)

RandomForestClassifier(oob_score=True)

### Score Checking 

In [32]:
y_pred = random_forest.predict(X_test)
print("Accuracy Score of Random Forest ::",accuracy_score(y_test,y_pred))

Accuracy Score of Random Forest :: 0.8524590163934426


### Out-of-Bag (OOB) Score

In [35]:
random_forest.oob_score_

0.7892561983471075

### Score Comparison 

In [38]:
test_accuracy = accuracy_score(y_test, y_pred)
oob_accuracy  = random_forest.oob_score_

print(f"Test Set Accuracy : {test_accuracy:.4f}")
print(f"OOB Score         : {oob_accuracy:.4f}")
print(f"Difference        : {abs(test_accuracy - oob_accuracy):.4f}")

# The OOB score is a free estimate — if it is close to the test accuracy,
# we can use it as a reliable proxy when a separate test set is unavailable

Test Set Accuracy : 0.8525
OOB Score         : 0.7893
Difference        : 0.0632
